# 🚀 Ollama en Google Colab con GPU T4 + Ngrok
### Proyecto: **Planos Históricos Organización**

Este notebook levanta **Ollama** con aceleración de GPU (NVIDIA T4 gratuita de Colab) y expone la API mediante un túnel de **Ngrok** para que el backend local (`fastapi` / `docker-compose`) procese planos de forma ultrarrápida (3 a 5 segundos por plano).

---  
### ⚠️ Requisito previo importante:
Verifica que este notebook esté corriendo con **GPU**:
👉 Menú superior: **Entorno de ejecución** -> **Cambiar tipo de entorno de ejecución** -> Seleccionar **GPU T4**.

In [ ]:
#@title 📦 1. Instalar dependencias, Ollama y pyngrok
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q pyngrok

---  
### 🔑 2. Iniciar Ollama y Túnel Ngrok
1. Ve a tu panel de Ngrok: [https://dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)
2. Copia tu **Authtoken** y pégalo en el campo `NGROK_TOKEN` a continuación.
3. Ejecuta la celda.

In [ ]:
#@title ⚙️ Iniciar Ollama y generar URL pública
!pip install -q pyngrok requests
import os
import subprocess
import time
import requests
from pyngrok import conf, installer, ngrok

NGROK_TOKEN = "TU_AUTHTOKEN_AQUI" #@param {type:"string"}

if NGROK_TOKEN == "TU_AUTHTOKEN_AQUI" or not NGROK_TOKEN.strip():
    raise ValueError("⚠️ Debes ingresar tu authtoken de ngrok antes de continuar.")

def asegurar_ollama_corriendo():
    """Verifica si Ollama responde en 127.0.0.1:11434; si no, lo inicia en una sesión independiente."""
    try:
        if requests.get("http://127.0.0.1:11434/api/tags", timeout=2).status_code == 200:
            print("✅ Servicio de Ollama en ejecución.")
            return
    except Exception:
        pass

    print("Iniciando servicio de Ollama en background (logs en /content/ollama.log)...")
    env = os.environ.copy()
    env["OLLAMA_HOST"] = "0.0.0.0:11434"
    env["OLLAMA_ORIGINS"] = "*"
    log_file = open("/content/ollama.log", "a")
    subprocess.Popen(
        ["ollama", "serve"],
        stdin=subprocess.DEVNULL,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        env=env,
        start_new_session=True,
    )
    for _ in range(15):
        time.sleep(1)
        try:
            if requests.get("http://127.0.0.1:11434/api/tags", timeout=2).status_code == 200:
                print("✅ Servicio de Ollama iniciado correctamente.")
                return
        except Exception:
            pass
    raise RuntimeError("No se pudo iniciar ollama serve.")

def asegurar_ngrok_corriendo(token: str) -> str:
    """Inicia el binario de ngrok en una sesión desacoplada para que Colab no lo cierre al terminar la celda."""
    try:
        res = requests.get("http://127.0.0.1:4040/api/tunnels", timeout=2).json()
        tunnels = res.get("tunnels", [])
        if tunnels:
            return tunnels[0]["public_url"]
    except Exception:
        pass

    ngrok_bin = conf.get_default().ngrok_path
    if not os.path.exists(ngrok_bin):
        installer.install_ngrok(ngrok_bin)
    ngrok.set_auth_token(token.strip())

    ngrok_log = open("/content/ngrok.log", "a")
    subprocess.Popen(
        [ngrok_bin, "http", "11434", "--log=stdout"],
        stdin=subprocess.DEVNULL,
        stdout=ngrok_log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    for _ in range(15):
        time.sleep(1)
        try:
            res = requests.get("http://127.0.0.1:4040/api/tunnels", timeout=2).json()
            tunnels = res.get("tunnels", [])
            if tunnels:
                return tunnels[0]["public_url"]
        except Exception:
            pass
    raise RuntimeError("No se pudo iniciar el túnel de Ngrok. Revisa /content/ngrok.log")

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
asegurar_ollama_corriendo()
public_url = asegurar_ngrok_corriendo(NGROK_TOKEN)

print("\n" + "=" * 65)
print("🚀 ¡OLLAMA ESTÁ CORRIENDO EN GOOGLE COLAB!")
print("Copia esta URL en tu archivo .env local:")
print(f"OLLAMA_HOST={public_url}")
print("=" * 65)

---  
### 📥 3. Seleccionar, descargar y activar el modelo de visión
Elige el modelo con el que deseas procesar los planos desde el menú desplegable (o escribe el nombre de cualquier modelo de visión de Ollama).  
Al ejecutar esta celda, el modelo se descargará y quedará marcado automáticamente como **modelo activo** (`colab-activo`). **No necesitas cambiar `VISION_MODEL` en tu archivo `.env` local**: el backend detectará automáticamente el modelo que elijas aquí. Puedes volver a ejecutar esta celda en cualquier momento para cambiar de modelo en caliente.

In [ ]:
#@title 🎯 Seleccionar y activar modelo de visión
import os
import subprocess
import time
import requests

MODELO = "qwen2.5vl:3b" #@param ["qwen2.5vl:3b", "qwen2.5vl:7b", "gemma3:4b", "gemma3:12b", "llama3.2-vision:11b"] {allow-input: true}
MODELO = MODELO.strip()

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"

# Verificar que el demonio de Ollama esté activo (y levantarlo si se detuvo)
try:
    requests.get("http://127.0.0.1:11434/api/tags", timeout=2).raise_for_status()
except Exception:
    print("⚙️ Iniciando demonio de Ollama...")
    env = os.environ.copy()
    env["OLLAMA_HOST"] = "0.0.0.0:11434"
    env["OLLAMA_ORIGINS"] = "*"
    log_file = open("/content/ollama.log", "a")
    subprocess.Popen(["ollama", "serve"], stdout=log_file, stderr=subprocess.STDOUT, env=env, start_new_session=True)
    for _ in range(15):
        time.sleep(1)
        try:
            if requests.get("http://127.0.0.1:11434/api/tags", timeout=2).status_code == 200:
                break
        except Exception:
            pass

print(f"📥 Descargando / verificando modelo '{MODELO}'...")
!ollama pull {MODELO}

# Registrar el modelo elegido bajo el alias 'colab-activo' para que el backend local lo detecte automáticamente
subprocess.run(["ollama", "cp", MODELO, "colab-activo"], check=True)

print("\n" + "=" * 65)
print(f"✅ Modelo activo configurado en Colab: {MODELO}")
print("El backend local usará automáticamente este modelo en el próximo plano.")
print("=" * 65)

---  
### 🩺 4. Verificar que el servicio responde a través del túnel

In [ ]:
#@title 🩺 Test de salud del túnel y modelo activo
import requests

try:
    if "asegurar_ollama_corriendo" in globals():
        asegurar_ollama_corriendo()
    if "asegurar_ngrok_corriendo" in globals() and "NGROK_TOKEN" in globals():
        public_url = asegurar_ngrok_corriendo(NGROK_TOKEN)
    res = requests.get(f"{public_url}/api/tags", headers={"ngrok-skip-browser-warning": "true"}, timeout=10)
    res.raise_for_status()
    models_data = res.json().get("models", [])
    modelos = [m["name"] for m in models_data if not m["name"].startswith("colab-activo")]
    activo_digest = next((m.get("digest") for m in models_data if m["name"].startswith("colab-activo")), None)
    modelo_activo = next((m["name"] for m in models_data if m.get("digest") == activo_digest and not m["name"].startswith("colab-activo")), MODELO if 'MODELO' in globals() else None)
    print(f"✅ Túnel activo y funcionando correctamente en: {public_url}")
    print(f"🎯 Modelo activo para el backend: {modelo_activo}")
    print(f"📦 Modelos descargados en Colab: {modelos}")
except Exception as e:
    print(f"❌ Error al verificar túnel: {e}")

---  
### 📊 5. Monitoreo de GPU (opcional)
Puedes ejecutar esta celda en cualquier momento para ver el consumo de VRAM y el estado de la GPU NVIDIA.

In [ ]:
!nvidia-smi

---  
### ⏱️ 6. Mantener la sesión activa mientras usas el programa local
Ejecuta esta celda para mantener el notebook corriendo indefinidamente mientras pruebas la API o la aplicación web local.

In [ ]:
#@title 📜 6. Ver logs de Ollama en vivo (y mantener sesión activa)
!tail -f /content/ollama.log